Task 1: PEAS Description

->Performance Measure: Picking up all packages successfully, avoiding flight in stormy weather (safety), minimizing delays, and accurately completing the route.

->Environment: A university campus divided into 6 discrete zones (0–5), containing packages and varying weather conditions.

->Actuators: Flight/Movement system (Advance/Hold), robotic arm/gripper (Pick up), and a communication/logging system (Report Route Complete/Reroute Alert).

->Sensors: Weather sensor (detects "Clear"/"Stormy"), Package sensor (detects "Package"/"Empty"), and a location sensor (current zone ID).

Task 2: Environment Classification

->Observable: Partially Observable. The drone can only sense the weather and package status of its current zone, remaining completely blind to the conditions of the other 5 zones.

->Deterministic: Stochastic. The environment can undergo unpredictable state changes, specifically regarding whether a zone will be "Clear" or "Stormy" at any given time.

->Static/Dynamic: Dynamic. The weather conditions (Stormy/Clear) can change while the drone is holding position or deliberating its next move.

->Discrete/Continuous: Discrete. The environment operates in clearly defined, separate states (6 specific zones, binary weather, binary package status) and time steps.

->Agents: Single-agent. There is only one delivery drone operating in this environment without interaction, competition, or cooperation with other AI entities.

In [1]:
class DroneEnvironment:
    def __init__(self):
        self.zones = [
            {"weather": "Clear",  "package": "Empty"},   # Zone 0
            {"weather": "Clear",  "package": "Package"}, # Zone 1
            {"weather": "Stormy", "package": "Empty"},   # Zone 2 (Stormy barrier)
            {"weather": "Clear",  "package": "Package"}, # Zone 3
            {"weather": "Stormy", "package": "Package"}, # Zone 4 (Stormy with package)
            {"weather": "Clear",  "package": "Empty"}    # Zone 5
        ]

    def get_percept(self, zone_idx):
        return self.zones[zone_idx]["weather"], self.zones[zone_idx]["package"]

    def pick_up_package(self, zone_idx):
        self.zones[zone_idx]["package"] = "Empty"

    def change_weather(self, zone_idx, new_weather):
        self.zones[zone_idx]["weather"] = new_weather


class SimpleReflexDroneAgent:
    def act(self, weather, package_status):
        if weather == "Stormy":
            return "Hold"
        elif weather == "Clear" and package_status == "Package":
            return "Pick Up"
        elif weather == "Clear" and package_status == "Empty":
            return "Advance"


class ModelBasedDroneAgent:
    def __init__(self):
        self.cleared_zones = set()
        self.consecutive_holds = 0

    def act(self, current_zone, weather, package_status):
        if weather == "Clear" and package_status == "Empty":
            self.cleared_zones.add(current_zone)
            
        if len(self.cleared_zones) == 6:
            return "Route Complete"

        if weather == "Stormy":
            self.consecutive_holds += 1
            if self.consecutive_holds > 3:
                return "Hold + Reroute Alert"
            return "Hold"

        self.consecutive_holds = 0

        if package_status == "Package":
            return "Pick Up"
        else:
            return "Advance"


print("--- TASK 3: Simple Reflex Agent Test (8 Steps) ---")
env1 = DroneEnvironment()
reflex_agent = SimpleReflexDroneAgent()
current_zone = 0

for step in range(1, 9):
    weather, package = env1.get_percept(current_zone)
    action = reflex_agent.act(weather, package)
    print(f"Step {step} | Zone {current_zone} | Percept: ({weather}, {package}) -> Action: {action}")
    
    if action == "Pick Up":
        env1.pick_up_package(current_zone)
    elif action == "Advance":
        current_zone = (current_zone + 1) % 6 # Move to next zone
        
print("\n--- TASK 4: Model-Based Agent Test (Run until Complete) ---")
env2 = DroneEnvironment()
model_agent = ModelBasedDroneAgent()
current_zone = 0
step = 1

while True:
    if step == 6:
        env2.change_weather(2, "Clear")
    if step == 11:
        env2.change_weather(4, "Clear")

    weather, package = env2.get_percept(current_zone)
    action = model_agent.act(current_zone, weather, package)
    print(f"Step {step:2} | Zone {current_zone} | Percept: ({weather:6}, {package:7}) | Action: {action}")
    
    if action == "Route Complete":
        break
    elif action == "Pick Up":
        env2.pick_up_package(current_zone)
    elif action == "Advance":
        current_zone = (current_zone + 1) % 6
        
    step += 1

--- TASK 3: Simple Reflex Agent Test (8 Steps) ---
Step 1 | Zone 0 | Percept: (Clear, Empty) -> Action: Advance
Step 2 | Zone 1 | Percept: (Clear, Package) -> Action: Pick Up
Step 3 | Zone 1 | Percept: (Clear, Empty) -> Action: Advance
Step 4 | Zone 2 | Percept: (Stormy, Empty) -> Action: Hold
Step 5 | Zone 2 | Percept: (Stormy, Empty) -> Action: Hold
Step 6 | Zone 2 | Percept: (Stormy, Empty) -> Action: Hold
Step 7 | Zone 2 | Percept: (Stormy, Empty) -> Action: Hold
Step 8 | Zone 2 | Percept: (Stormy, Empty) -> Action: Hold

--- TASK 4: Model-Based Agent Test (Run until Complete) ---
Step  1 | Zone 0 | Percept: (Clear , Empty  ) | Action: Advance
Step  2 | Zone 1 | Percept: (Clear , Package) | Action: Pick Up
Step  3 | Zone 1 | Percept: (Clear , Empty  ) | Action: Advance
Step  4 | Zone 2 | Percept: (Stormy, Empty  ) | Action: Hold
Step  5 | Zone 2 | Percept: (Stormy, Empty  ) | Action: Hold
Step  6 | Zone 2 | Percept: (Clear , Empty  ) | Action: Advance
Step  7 | Zone 3 | Percept: (C

Task 5: Agent Comparison and Structural Limitations

The two agents will behave distinctly differently if placed in a "Stormy" zone for four consecutive steps, as well as at the end of their route. The Model-Based agent will output a "Reroute Alert" on the fourth step due to its internal counter, and will successfully halt and output "Route Complete" once all zones are clear. The Simple Reflex Agent, conversely, will silently output "Hold" indefinitely in the storm, and would blindly attempt to advance off the edge of the map if it cleared the final zone. Rule 5 is structurally impossible for a Simple Reflex Agent because it maps actions exclusively to the instantaneous, current percept (the immediate weather and package status). It possesses no memory or internal state to track historical data, meaning it cannot count how many times it has previously chosen to hold position. Without the ability to store a sequence of past events, it cannot evaluate any condition that requires knowledge of time duration or past actions.